# GPT-2 Fine-Tuning — Emily Prime Corpus

Runs on Google Colab (T4 GPU). Fine-tunes GPT-2 small (124M) on the Emily Prime
training corpus built by `scripts/prime_directive_dataset.py`.

**Open this directly from GitHub — no upload needed:** Colab → File → Open notebook →
GitHub tab → `emilyspringerton/gpt2-alpine-c` → `notebooks/gpt2_finetune_colab.ipynb`.

**Why Colab, not local:** confirmed 2026-07-17 — the dev VM's FatBaby pipeline processes consume
~3GB RSS on a 3.8GB box, leaving no headroom for even the memory-conscious local LoRA trainer
(`scripts/train_local.py`); two attempts were silently OOM-killed within seconds of starting.
Full runbook with corpus-upload options: `docs/COLAB_RUNBOOK.md`.

**Paste-once workflow (added 2026-07-17):** the one code cell below is all you ever paste into
Colab. Hit play, approve the Drive OAuth prompt when it appears, and it handles everything —
clones (or pulls) `gpt2-alpine-c`, then runs `scripts/colab_train.py` for the actual training.
All training logic lives in that script, in git, not in this notebook — when the training
approach changes, it ships as a commit, and the *same* bootstrap cell picks it up on the next
run via `git pull`. Nothing to re-paste, no cells to manually resync.

**Expected training time (T4):** ~20-40 min for 1000 steps on ~5MB corpus. Current corpus
(2026-07-17 build) is 1048 records / 1.3MB — well inside that.

**Target:** entropy delta ≥ 0.5 nats over base GPT-2 (base H_mean=4.4877 nats). A 300-step
local CPU run (2026-06-23) only reached +0.17 nats — this full Colab T4 run is what's needed to
close the gap (NORTHSTAR.md Milestone 3).


In [ ]:
# === Emily Prime GPT-2 fine-tune — reusable bootstrap cell ===
# This cell is the only thing you ever need to paste into Colab. It mounts
# Drive (approve the OAuth prompt when it appears), then pulls the latest
# training logic from git and runs it. Future changes to how training works
# ship as commits to scripts/colab_train.py — re-running this same cell
# always executes the current version, no re-pasting required.

from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess

REPO_URL = 'https://github.com/emilyspringerton/gpt2-alpine-c.git'
REPO_DIR = '/content/gpt2-alpine-c'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# Adjust DRIVE_FOLDER here only if your Drive layout differs from the default.
os.environ.setdefault('DRIVE_FOLDER', '/content/drive/MyDrive/emily-training')

subprocess.run(
    ['python3', 'scripts/colab_train.py'],
    cwd=REPO_DIR, check=True,
)


## Next Steps

After training:
1. Download `checkpoint-final.tar.gz` from Drive
2. Convert to C binary:
   ```bash
   python3 scripts/convert_ft_checkpoint.py \
     --checkpoint checkpoint-final.tar.gz \
     --output weights/emily-ft.bin
   ```
3. Test entropy:
   ```bash
   ./gpt2_run weights/emily-ft.bin --entropy-stats
   ```
4. File a completion Apple:
   ```bash
   emily apples post -t completion "GPT-2 Emily fine-tune complete" "..."
   ```
5. Mark S26-02 done in EMILY/BACKLOG.md